# F1 Model Performance Analysis

This notebook provides a comprehensive analysis of our three YOLOv11 models' performance:
1. F1 Car Detection
2. F1 Team Detection
3. F1 Driver Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import torch
import cv2
from PIL import Image
import json
from sklearn.metrics import confusion_matrix
import time

## 1. Model Loading and Setup

Load our three trained models for comparative analysis.

In [ ]:
# Load models
car_model = YOLO('models/car_detection/best.pt')
team_model = YOLO('models/team_detection/best.pt')
driver_model = YOLO('models/driver_detection/best.pt')

models = {
    'Car Detection': car_model,
    'Team Detection': team_model,
    'Driver Detection': driver_model
}

## 2. Comparative Performance Metrics

Compare key metrics across all three models:

In [ ]:
def get_model_metrics(model, test_data):
    results = model.val(data=test_data)
    metrics = results.results_dict
    return {
        'mAP50': metrics['metrics/mAP50(B)'],
        'mAP50-95': metrics['metrics/mAP50-95(B)'],
        'Precision': metrics['metrics/precision(B)'],
        'Recall': metrics['metrics/recall(B)'],
        'Speed (ms)': results.speed['inference']
    }

model_metrics = {}
for name, model in models.items():
    model_metrics[name] = get_model_metrics(model, f'data/{name.lower().replace(" ", "_")}.yaml')

metrics_df = pd.DataFrame(model_metrics).T

# Plot comparative metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
metrics = ['mAP50', 'mAP50-95', 'Precision', 'Recall']
for ax, metric in zip(axes.flat, metrics):
    sns.barplot(data=metrics_df, y=metrics_df.index, x=metric, ax=ax)
    ax.set_title(f'Comparison of {metric}')
plt.tight_layout()
plt.show()

## 3. Per-Class Analysis

In [ ]:
def analyze_per_class_performance(model, class_names):
    results = model.val(data='data/test')
    per_class_metrics = {}
    
    for i, name in enumerate(class_names):
        per_class_metrics[name] = {
            'Precision': results.results_dict[f'metrics/precision(B)_{i}'],
            'Recall': results.results_dict[f'metrics/recall(B)_{i}'],
            'mAP50': results.results_dict[f'metrics/mAP50(B)_{i}']
        }
    
    return pd.DataFrame(per_class_metrics).T

# Example for driver detection model
driver_class_names = ['RedBull_Ver', 'RedBull_Per', 'Ferrari_Lec', ...] # Add all driver names
driver_per_class = analyze_per_class_performance(driver_model, driver_class_names)

plt.figure(figsize=(15, 8))
sns.heatmap(driver_per_class, annot=True, cmap='YlOrRd', fmt='.2f')
plt.title('Per-Driver Performance Metrics')
plt.tight_layout()
plt.show()

## 4. Speed and Efficiency Analysis

In [ ]:
def benchmark_model(model, test_images, batch_sizes=[1, 4, 8, 16]):
    benchmarks = []
    for batch_size in batch_sizes:
        times = []
        for _ in range(5):  # 5 runs for average
            start = time.time()
            _ = model(test_images, batch=batch_size)
            times.append((time.time() - start) / len(test_images))
        benchmarks.append({
            'batch_size': batch_size,
            'avg_time': np.mean(times),
            'std_time': np.std(times)
        })
    return pd.DataFrame(benchmarks)

# Plot speed comparisons
plt.figure(figsize=(10, 6))
for name, model in models.items():
    benchmarks = benchmark_model(model, test_images)
    plt.errorbar(benchmarks['batch_size'], benchmarks['avg_time'],
                yerr=benchmarks['std_time'], label=name, marker='o')

plt.xlabel('Batch Size')
plt.ylabel('Time per Image (s)')
plt.title('Model Speed vs Batch Size')
plt.legend()
plt.grid(True)
plt.show()

## 5. Error Analysis

In [ ]:
def analyze_errors(model, test_images, ground_truth):
    errors = []
    for img, gt in zip(test_images, ground_truth):
        results = model(img)
        pred_boxes = results[0].boxes.xyxy
        pred_conf = results[0].boxes.conf
        pred_cls = results[0].boxes.cls
        
        # Calculate IoU and find mismatches
        for gt_box in gt['boxes']:
            ious = box_iou(gt_box, pred_boxes)
            if max(ious) < 0.5:  # Missed detection
                errors.append({
                    'type': 'missed_detection',
                    'image': img,
                    'gt_box': gt_box
                })
    return errors

# Visualize some error cases
def plot_error_cases(errors, num_cases=5):
    fig, axes = plt.subplots(1, num_cases, figsize=(20, 4))
    for ax, error in zip(axes, errors[:num_cases]):
        img = cv2.imread(error['image'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        box = error['gt_box']
        rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                           fill=False, color='red')
        ax.add_patch(rect)
        ax.axis('off')
        ax.set_title(error['type'])
    plt.tight_layout()
    plt.show()

## 6. Performance in Different Conditions

In [ ]:
def analyze_conditions(model, test_data):
    conditions = {
        'daytime': test_data['daytime_images'],
        'nighttime': test_data['nighttime_images'],
        'rain': test_data['rain_images'],
        'close_up': test_data['close_up_images'],
        'distant': test_data['distant_images']
    }
    
    results = {}
    for condition, images in conditions.items():
        metrics = model.val(data=images)
        results[condition] = {
            'mAP50': metrics.results_dict['metrics/mAP50(B)'],
            'Precision': metrics.results_dict['metrics/precision(B)'],
            'Recall': metrics.results_dict['metrics/recall(B)']
        }
    
    return pd.DataFrame(results).T

# Plot performance across conditions
plt.figure(figsize=(12, 6))
for name, model in models.items():
    results = analyze_conditions(model, test_data)
    plt.subplot(131)
    plt.title('mAP50 by Condition')
    plt.plot(results.index, results['mAP50'], label=name, marker='o')
    plt.xticks(rotation=45)
    plt.legend()

plt.tight_layout()
plt.show()